**Cell 1: Setup & Environment**

In [2]:
!pip install -q python-dotenv tqdm tenacity datasets aiohttp aiofiles

import os
import json
import re
import asyncio
import aiohttp
import aiofiles
import pandas as pd
from tqdm.asyncio import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from datasets import load_dataset

# ==========================================
# XỬ LÝ BẢO MẬT API KEY 
# ==========================================
try:
    # Thử lấy từ Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")
    print("Đã tải API Key từ Kaggle Secrets.")
except ImportError:
    try:
        # Fallback nếu chạy Local trên máy cá nhân
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        print("Đã tải API Key từ file .env (Local).")
    except:
        OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    raise ValueError("Không tìm thấy OPENROUTER_API_KEY. Vui lòng thiết lập trong mục Add-ons -> Secrets của Kaggle.")

Đã tải API Key từ Kaggle Secrets.


**Cell 2: Configurations & Prompts**

In [3]:
# ==========================================
# CẤU HÌNH DỮ LIỆU & HỆ THỐNG
# ==========================================
HF_DATASET_NAME = ""
HF_SPLIT = "train"

# KAGGLE INPUT PATH
FILE_PATH = "/kaggle/input/datasets/quangminh2401/medthink-benchfull/QA_data_301_400.json" 
SAMPLE_TEXT = {}

# CÁC CỘT CẦN DỊCH
COLUMNS_TO_TRANSLATE = ["QA_Type", "question", "answer", "Scoring_Points"]
CONCURRENT_REQUESTS = 10 
REQUEST_TIMEOUT_SECONDS = 120

# ------------------------------------------
# LOGIC TẠO TÊN FOLDER & FILE (Đã ép vào /kaggle/working/)
# ------------------------------------------
if FILE_PATH and os.path.exists(FILE_PATH):
    base_name = os.path.splitext(os.path.basename(FILE_PATH))[0]
elif HF_DATASET_NAME:
    base_name = HF_DATASET_NAME.replace("/", "_")
else:
    base_name = "sample_test"

# KAGGLE OUTPUT DIR: Bắt buộc phải lưu vào /kaggle/working/
KAGGLE_WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(KAGGLE_WORKING_DIR, f"{base_name}_result")
os.makedirs(OUTPUT_DIR, exist_ok=True) 

# File chạy ngầm (JSONL)
OUTPUT_JSONL_PATH = os.path.join(OUTPUT_DIR, f"vi_{base_name}.jsonl")
EVAL_LOG_JSONL_PATH = os.path.join(OUTPUT_DIR, f"eval_{base_name}.jsonl")

# File thành phẩm cuối cùng (Exports)
FINAL_JSON_PATH = os.path.join(OUTPUT_DIR, f"vi_{base_name}.json")
FINAL_CSV_PATH = os.path.join(OUTPUT_DIR, f"vi_{base_name}.csv")
FINAL_PARQUET_PATH = os.path.join(OUTPUT_DIR, f"vi_{base_name}.parquet")

FINAL_EVAL_CSV_PATH = os.path.join(OUTPUT_DIR, f"eval_{base_name}.csv")
FINAL_EVAL_PARQUET_PATH = os.path.join(OUTPUT_DIR, f"eval_{base_name}.parquet")
FINAL_EVAL_JSON_PATH = os.path.join(OUTPUT_DIR, f"eval_{base_name}.json")

# ==========================================
# CẤU HÌNH MODEL & PROMPT
# ==========================================
MODEL_DRAFT = "openai/gpt-5.4"
MODEL_REFINE = "anthropic/claude-sonnet-4.6"

SYSTEM_PROMPT_DRAFT = """
Bạn là một bác sĩ lâm sàng đồng thời là biên dịch viên y khoa chuyên nghiệp (Anh sang Việt).
Nhiệm vụ của bạn là dịch các giá trị (values) trong một đối tượng JSON từ Tiếng Anh sang Tiếng Việt.

YÊU CẦU BẮT BUỘC:

1. BẢO TOÀN CẤU TRÚC
- Giữ nguyên tuyệt đối các khóa (keys), mảng (arrays) và cấu trúc lồng nhau của JSON
- Chỉ dịch nội dung text bên trong các values

RÀNG BUỘC NGHIÊM NGẶT:
- Keys phải giữ nguyên 100 phần trăm, không đổi tên, không dịch
- Không thêm key mới
- Không xóa key
- Thứ tự keys phải giữ nguyên như bản gốc
- Kiểu dữ liệu phải giữ nguyên (string, number, array, object)

2. BẢO TOÀN NGHĨA (QUAN TRỌNG NHẤT)
- Dịch tương đương một-một với câu gốc
- Không thêm, không bớt, không suy diễn

3. THUẬT NGỮ Y KHOA
- Nếu biết thuật ngữ tiếng Việt chuẩn thì sử dụng chính xác thuật ngữ đó
- Nếu không chắc chắn thì giữ nguyên thuật ngữ tiếng Anh
- Không dịch từng từ đối với thuật ngữ
- Không tự tạo thuật ngữ hoặc dùng mô tả gần nghĩa

4. KHÔNG SUY DIỄN
- Không thêm bất kỳ thông tin nào không có trong bản gốc
- Không thêm vị trí giải phẫu, mức độ, mục đích hoặc diễn giải

5. ĐỘ CHÍNH XÁC NGỮ NGHĨA
- Giữ đúng các yếu tố:
  + so sánh (higher, lower, lowest, most, least)
  + phủ định (no, not, without, exclude)
  + điều kiện (before, after)

6. CHẨN ĐOÁN VÀ SỐ LIỆU
- Giữ nguyên tuyệt đối liều lượng, đơn vị đo (mg, mmol/L...), định dạng số

OUTPUT FORMAT:
- Trả về đúng một đối tượng JSON hợp lệ
- Giữ nguyên cấu trúc JSON, keys tương ứng một-một với bản gốc
- Không kèm markdown, không giải thích thêm
"""

USER_PROMPT_DRAFT_TEMPLATE = """
Dịch các values trong JSON sau sang tiếng Việt:
{json_data}
"""

SYSTEM_PROMPT_REFINE = """
Bạn là một chuyên gia đánh giá bản dịch y khoa cấp cao (Anh sang Việt), có kiến thức sâu về:
- thuật ngữ lâm sàng
- dược lý học
- viết học thuật trong y sinh

Nhiệm vụ của bạn là:
1. Đối chiếu JSON bản gốc (tiếng Anh) với JSON bản dịch (Draft tiếng Việt)
2. Đánh giá chất lượng bản dịch với mức độ khắt khe cao
3. Đảm bảo tính chính xác tuyệt đối và an toàn cho người bệnh
4. Tạo ra một bản JSON hoàn thiện (refined_json) tốt nhất

NGUYÊN TẮC ĐÁNH GIÁ:

- Ưu tiên cao nhất là độ chính xác y khoa và tính trung thực với bản gốc
- Không được thêm hoặc suy diễn bất kỳ thông tin nào không có trong source_json

Đặc biệt chú ý:
- Sai thuật ngữ chuyên ngành
- Sai liều lượng, đơn vị, chỉ định
- Sai nghĩa liên quan đến so sánh, phủ định, điều kiện

Bất kỳ lỗi nào có thể ảnh hưởng đến an toàn bệnh nhân thì đặt critical_error = true

NGUYÊN TẮC CHỈNH SỬA:

- Nếu bản dịch draft đã đúng thì giữ nguyên
- Chỉ sửa khi có một trong các lỗi sau:
  + sai nghĩa
  + sai thuật ngữ
  + sai dữ kiện
- Không viết lại câu nếu không cần thiết
- Không thay đổi cách diễn đạt chỉ để mượt hơn

THUẬT NGỮ:

- Sử dụng thuật ngữ tiếng Việt chuẩn nếu chắc chắn
- Nếu không chắc chắn thì giữ nguyên tiếng Anh
- Không thay bằng mô tả gần nghĩa

TIÊU CHÍ CHẤM ĐIỂM (0–10 mỗi tiêu chí):
- accuracy: độ chính xác nội dung y khoa
- terminology: sử dụng thuật ngữ chuyên ngành
- fluency: độ tự nhiên, dễ hiểu
- completeness: đầy đủ nội dung

CÁCH TÍNH ĐIỂM:
- overall_score là trung bình của 4 tiêu chí

YÊU CẦU QUAN TRỌNG:

- refined_json phải giữ cấu trúc keys một-một với source_json

RÀNG BUỘC NGHIÊM NGẶT:
- Không thêm, không xóa, không đổi tên bất kỳ key nào
- Thứ tự keys phải giống source_json
- Kiểu dữ liệu phải giữ nguyên
- Mọi value phải tương ứng đúng vị trí với source_json

- Nếu draft_json sai cấu trúc thì phải sửa lại cho đúng với source_json

OUTPUT FORMAT (JSON DUY NHẤT):
{
  "overall_score": float (0-10),
  "scores": {
    "accuracy": int (0-10),
    "terminology": int (0-10),
    "fluency": int (0-10),
    "completeness": int (0-10)
  },
  "critical_error": true/false,
  "issues": ["mô tả lỗi cụ thể, chi tiết"],
  "summary": "nhận xét tổng quan, mang tính chuyên môn cao",
  "refined_json": { ... }
}
"""

USER_PROMPT_REFINE_TEMPLATE = """
NHIỆM VỤ:
Đánh giá và cải thiện bản dịch JSON tiếng Việt của nội dung y khoa.

JSON GỐC (TIẾNG Anh):
{source_json}

JSON BẢN DỊCH DRAFT (TIẾNG VIỆT):
{draft_json}

YÊU CẦU:
- Đánh giá cực kỳ nghiêm khắc theo tiêu chuẩn y khoa
- Chỉ ra lỗi chi tiết, đặc biệt các lỗi có nguy cơ lâm sàng
- Cải thiện bản dịch để đạt chất lượng cao nhất
"""

**Cell 3 & 4: Pipeline Logic & Data Loaders**


In [4]:
class QuotaExceededError(Exception):
    pass

def extract_json_from_llm(text):
    text = str(text).strip()
    text = re.sub(r'^```json\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'^```\s*', '', text, flags=re.MULTILINE)
    try: return json.loads(text)
    except:
        match = re.search(r'(\{.*\}|\[.*\])', text, re.DOTALL)
        if match:
            try: return json.loads(match.group(0))
            except: pass
        raise ValueError("Failed to parse JSON.")

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=20), retry=retry_if_exception_type((aiohttp.ClientError, ValueError, asyncio.TimeoutError, KeyError)))
async def call_openrouter_async(session, model_id, system_prompt, user_prompt, state, require_json=False):
    # ... (Giữ nguyên phần headers và payload) ...
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    payload = {"model": model_id, "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]}
    if require_json: payload["response_format"] = {"type": "json_object"}

    async with session.post("https://openrouter.ai/api/v1/chat/completions",
        headers=headers, json=payload,
        timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_SECONDS, connect=10, sock_read=100)
    ) as response:
        if response.status != 200: 
            error_text = await response.text()
            if response.status == 429 and ("credits" in error_text.lower() or "free-models" in error_text.lower() or "limit" in error_text.lower()):
                raise QuotaExceededError(error_text)
            raise ValueError(f"API Error {response.status}: {error_text}")
            
        data = await response.json()
        usage = data.get("usage", {})
        
        # CẬP NHẬT STATS VÀO BIẾN STATE ĐƯỢC TRUYỀN VÀO
        state['token_stats'][model_id]["prompt"] += usage.get("prompt_tokens", 0)
        state['token_stats'][model_id]["completion"] += usage.get("completion_tokens", 0)
        state['token_stats'][model_id]["calls"] += 1
        
        try: content = data['choices'][0]['message']['content']
        except: raise ValueError("API missing 'choices'")
        if content is None or str(content).strip() == "": raise ValueError("API null content")
        return str(content).strip()

async def async_medical_translation_pipeline(session, source_dict, state):
    if not source_dict: return source_dict, None
    source_json_str = json.dumps(source_dict, ensure_ascii=False)

    try:
        user_prompt_draft = USER_PROMPT_DRAFT_TEMPLATE.format(json_data=source_json_str)
        # Truyền state vào
        draft_str = await call_openrouter_async(session, MODEL_DRAFT, SYSTEM_PROMPT_DRAFT, user_prompt_draft, state, require_json=True)
        draft_dict = extract_json_from_llm(draft_str)
        if not isinstance(draft_dict, dict): raise ValueError("Draft not a dict")
    except QuotaExceededError:
        state['quota_exceeded'] = True # Bật cờ lỗi vào state
        raise 
    except Exception as e:
        state['error_stats']["draft_errors"] += 1
        if len(state['error_stats']["error_details"]) < 5: 
            state['error_stats']["error_details"].append(f"Draft Error: {str(e)[:100]}...")
        return None, None 

    try:
        user_prompt_refine = USER_PROMPT_REFINE_TEMPLATE.format(source_json=source_json_str, draft_json=json.dumps(draft_dict, ensure_ascii=False))
        # Truyền state vào
        refine_str = await call_openrouter_async(session, MODEL_REFINE, SYSTEM_PROMPT_REFINE, user_prompt_refine, state, require_json=True)
        eval_data = extract_json_from_llm(refine_str)
        refined_dict = eval_data.get("refined_json")
        if not isinstance(refined_dict, dict): raise ValueError("Refine missing 'refined_json'")
        if "refined_json" in eval_data: del eval_data["refined_json"]
        eval_data["_source"] = source_dict
        eval_data["_draft"] = draft_dict
    except QuotaExceededError:
        state['quota_exceeded'] = True
        raise 
    except Exception as e:
        state['error_stats']["refine_errors"] += 1
        if len(state['error_stats']["error_details"]) < 5: 
            state['error_stats']["error_details"].append(f"Refine Error: {str(e)[:100]}...")
        return None, None

    return refined_dict, eval_data

def get_input_data():
    if HF_DATASET_NAME != "": return load_dataset(HF_DATASET_NAME, split=HF_SPLIT).to_pandas().to_dict(orient='records')
    elif FILE_PATH != "":
        ext = FILE_PATH.split('.')[-1].lower()
        if ext == 'json':
            with open(FILE_PATH, 'r', encoding='utf-8') as f:
                d = json.load(f)
                return d if isinstance(d, list) else [d]
        elif ext == 'jsonl':
            d = []
            with open(FILE_PATH, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip(): d.append(json.loads(line))
            return d
        elif ext == 'csv': return pd.read_csv(FILE_PATH).fillna("").to_dict(orient='records')
        elif ext == 'parquet': return pd.read_parquet(FILE_PATH).fillna("").to_dict(orient='records')
    return None

def extract_subset_for_translation(record, columns):
    subset = {}
    dropped = []

    for k in columns:
        if k not in record or record[k] is None:
            dropped.append(f"{k}=<missing>")
            continue
        val = record[k]
        if isinstance(val, (dict, list)):
            if len(val) > 0:
                subset[k] = val
            else:
                dropped.append(f"{k}=<empty_collection>")
        elif isinstance(val, (int, float, bool)):
            subset[k] = val
        elif isinstance(val, str):
            if val.strip() != "":
                subset[k] = val
            else:
                dropped.append(f"{k}=<empty_string>")

    if dropped:
        sidx = record.get("_sidx", "?")
        tqdm.write(f"[WARN] _sidx={sidx}: {len(dropped)} column(s) bị drop "
                   f"(giữ nguyên tiếng Anh): {dropped}")

    return subset

def load_processed_ids(jsonl_path):
    processed = set()
    corrupt_lines = 0

    if not os.path.exists(jsonl_path):
        return processed

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            if "_sidx" in record:
                processed.add(record["_sidx"])
        except json.JSONDecodeError:
            corrupt_lines += 1
            print(f"[WARN] Dòng {i+1} trong JSONL bị corrupt, bỏ qua. "
                  f"Nội dung: {line[:80]!r}...")

    if corrupt_lines > 0:
        import shutil
        backup_path = jsonl_path + ".backup"
        shutil.copy2(jsonl_path, backup_path)
        print(f"[INFO] Đã backup JSONL bị corrupt → {backup_path} "
              f"({corrupt_lines} dòng lỗi)")

    return processed

**Cell 5: Execution & Display Summary**

In [5]:
async def worker_task(worker_id, queue, session, state, out_f, eval_f, pbar):
    while True:
        try:
            # Lấy 1 record ra từ hàng đợi (Không block nếu queue trống)
            record = queue.get_nowait()
        except asyncio.QueueEmpty:
            # Nếu queue hết việc, công nhân nghỉ hưu
            break

        # Nếu có công nhân khác báo lỗi Quota, công nhân này cũng dừng làm việc ngay
        if state['quota_exceeded']:
            queue.task_done()
            continue

        try:
            subset_to_translate = extract_subset_for_translation(record, COLUMNS_TO_TRANSLATE)
            if not subset_to_translate:
                continue

            refined_subset, eval_data = await async_medical_translation_pipeline(
                session, subset_to_translate, state
            )
            
            if refined_subset is None:
                continue

            # Merge keys
            expected_keys = set(subset_to_translate.keys())
            translated_record = record.copy()
            for k in expected_keys:
                if k in refined_subset:
                    translated_record[k] = refined_subset[k]

            if eval_data:
                eval_data["_sidx"] = record["_sidx"]
            
            # Ghi vào file
            await out_f.write(json.dumps(translated_record, ensure_ascii=False) + '\n')
            if eval_data: 
                await eval_f.write(json.dumps(eval_data, ensure_ascii=False) + '\n')
                
        except QuotaExceededError:
            tqdm.write(f"[Worker {worker_id}] Gặp lỗi Quota. Ra hiệu dừng hệ thống!")
            state['quota_exceeded'] = True
        except Exception as e:
            tqdm.write(f"[Worker {worker_id}] Lỗi không xác định: {e}")
        finally:
            # Báo cho Queue biết là record này đã được xử lý xong
            queue.task_done()
            pbar.update(1)

async def main():
    # 1. KHỞI TẠO STATE SẠCH (Khắc phục lỗi chạy lại cell của Jupyter)
    state = {
        'quota_exceeded': False,
        'token_stats': {MODEL_DRAFT: {"prompt": 0, "completion": 0, "calls": 0}, MODEL_REFINE: {"prompt": 0, "completion": 0, "calls": 0}},
        'error_stats': {"draft_errors": 0, "refine_errors": 0, "error_details": []}
    }

    # 2. XỬ LÝ DỮ LIỆU ĐẦU VÀO
    data_records = get_input_data()
    if data_records is None: data_records = [SAMPLE_TEXT]

    for i, record in enumerate(data_records): record["_sidx"] = i
    processed_sidx = load_processed_ids(OUTPUT_JSONL_PATH)
    pending_records = [r for r in data_records if r["_sidx"] not in processed_sidx]
    
    print(f"Tổng: {len(data_records)} | Đã xong: {len(processed_sidx)} | Cần xử lý: {len(pending_records)}")
    if not pending_records: return

    # 3. NẠP DỮ LIỆU VÀO QUEUE (Băng chuyền)
    queue = asyncio.Queue()
    for record in pending_records:
        queue.put_nowait(record)

    # 4. TẠO WORKERS VÀ THỰC THI
    async with aiohttp.ClientSession() as session:
        async with aiofiles.open(OUTPUT_JSONL_PATH, mode='a', encoding='utf-8') as out_f, \
                   aiofiles.open(EVAL_LOG_JSONL_PATH, mode='a', encoding='utf-8') as eval_f:
            
            with tqdm(total=len(pending_records), desc="Đang dịch") as pbar:
                # Tuyển dụng công nhân dựa trên CONCURRENT_REQUESTS (ví dụ: 1)
                workers = [
                    asyncio.create_task(worker_task(i, queue, session, state, out_f, eval_f, pbar))
                    for i in range(CONCURRENT_REQUESTS)
                ]
                
                # Quản đốc đứng đợi băng chuyền (queue) xử lý xong hết
                await queue.join()

    # 5. BÁO CÁO KẾT QUẢ CUỐI CÙNG
    if state['quota_exceeded']:
        print("\n[CRITICAL] Dừng khẩn cấp: Chạm giới hạn quota API.")
        print("[INFO] Đã flush checkpoint. Resume lần sau sẽ tiếp tục từ mẫu lỗi.")

    print("\n" + "="*50)
    print("BÁO CÁO HỆ THỐNG (SYSTEM SUMMARY)")
    print("="*50)
    print("\n1. SỬ DỤNG TOKEN (Token Usage):")
    for model, stats in state['token_stats'].items():
        print(f"   [{model}] API Calls: {stats['calls']} | Prompt: {stats['prompt']:,} | Completion: {stats['completion']:,}")

    print("\n2. THỐNG KÊ LỖI (Error Statistics):")
    total_errors = state['error_stats']["draft_errors"] + state['error_stats']["refine_errors"]
    if total_errors == 0:
        print("      Không có mẫu nào bị lỗi (Skip).")
    else:
        print(f"      Có tổng cộng {total_errors} mẫu bị bỏ qua (Skip).")
        print(f"      - Lỗi ở bước Draft: {state['error_stats']['draft_errors']} mẫu")
        print(f"      - Lỗi ở bước Refine: {state['error_stats']['refine_errors']} mẫu")
        
        if state['error_stats']["error_details"]:
            print("\n   Một số lỗi tiêu biểu (Debug Info):")
            for err in state['error_stats']["error_details"]:
                print(f"      - {err}")
    print("="*50)

# Kích hoạt chương trình
try:
    await main()
except KeyboardInterrupt:
    print("NGƯỜI DÙNG CHỦ ĐỘNG DỪNG CHƯƠNG TRÌNH (Keyboard Interrupt)!")

Tổng: 100 | Đã xong: 92 | Cần xử lý: 8


Đang dịch: 100%|██████████| 8/8 [00:16<00:00,  2.12s/it]


BÁO CÁO HỆ THỐNG (SYSTEM SUMMARY)

1. SỬ DỤNG TOKEN (Token Usage):
   [openai/gpt-5.4] API Calls: 0 | Prompt: 0 | Completion: 0
   [anthropic/claude-sonnet-4.6] API Calls: 0 | Prompt: 0 | Completion: 0

2. THỐNG KÊ LỖI (Error Statistics):
      Có tổng cộng 8 mẫu bị bỏ qua (Skip).
      - Lỗi ở bước Draft: 8 mẫu
      - Lỗi ở bước Refine: 0 mẫu

   Một số lỗi tiêu biểu (Debug Info):
      - Draft Error: RetryError[<Future at 0x78209da6fbc0 state=finished raised ValueError>]...
      - Draft Error: RetryError[<Future at 0x78209da6f7d0 state=finished raised ValueError>]...
      - Draft Error: RetryError[<Future at 0x78209dacea50 state=finished raised ValueError>]...
      - Draft Error: RetryError[<Future at 0x78209da6ef30 state=finished raised ValueError>]...
      - Draft Error: RetryError[<Future at 0x7820ed5e9340 state=finished raised ValueError>]...


**Cell 6: Convert JSONL -> JSON/CSV/PARQUET**

In [17]:
import pandas as pd
import json
import os

def auto_sort_dataframe(df):
    if "_sidx" in df.columns:
        df = df.sort_values(by="_sidx").reset_index(drop=True)
        df = df.drop(columns=["_sidx"]) 
    return df

def convert_jsonl_to_json(jsonl_path, final_json_path):
    try:
        if os.path.exists(jsonl_path):
            data = [json.loads(line) for line in open(jsonl_path, 'r', encoding='utf-8') if line.strip()]
            df = auto_sort_dataframe(pd.DataFrame(data))
            with open(final_json_path, 'w', encoding='utf-8') as f:
                json.dump(df.to_dict(orient='records'), f, ensure_ascii=False, indent=4)
            print(f"Đã tạo: {final_json_path}")
    except Exception as e: print(f"Lỗi: {e}")

def convert_jsonl_to_csv(jsonl_path, final_csv_path):
    try:
        if os.path.exists(jsonl_path):
            data = [json.loads(line) for line in open(jsonl_path, 'r', encoding='utf-8') if line.strip()]
            df = auto_sort_dataframe(pd.DataFrame(data))
            df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
            print(f"Đã tạo: {final_csv_path}")
    except Exception as e: print(f"Lỗi: {e}")

def convert_jsonl_to_parquet(jsonl_path, final_parquet_path):
    try:
        if os.path.exists(jsonl_path):
            data = [json.loads(line) for line in open(jsonl_path, 'r', encoding='utf-8') if line.strip()]
            df = auto_sort_dataframe(pd.DataFrame(data))
            for col in df.columns:
                if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                    df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else x)
            df.to_parquet(final_parquet_path, engine='pyarrow', index=False)
            print(f"Đã tạo: {final_parquet_path}")
    except Exception as e: print(f"Lỗi: {e}")

# Kích hoạt các hàm xuất file bạn cần:
# convert_jsonl_to_json(OUTPUT_JSONL_PATH, FINAL_JSON_PATH)
convert_jsonl_to_csv(OUTPUT_JSONL_PATH, FINAL_CSV_PATH)
# convert_jsonl_to_parquet(OUTPUT_JSONL_PATH, FINAL_PARQUET_PATH)

convert_jsonl_to_csv(EVAL_LOG_JSONL_PATH, FINAL_EVAL_CSV_PATH)

Đã tạo: /kaggle/working/QA_data_301_400_result/vi_QA_data_301_400.csv
Đã tạo: /kaggle/working/QA_data_301_400_result/eval_QA_data_301_400.csv


**Debugging Section**

**Check mẫu đang bị kẹt**

In [18]:
import json

# Lấy lại toàn bộ data
data_records = get_input_data()
for i, record in enumerate(data_records): record["_sidx"] = i

# Lấy danh sách ID đã thành công
processed_sidx = load_processed_ids(OUTPUT_JSONL_PATH)

# Trích xuất đúng 8 mẫu thất bại
failed_records = [r for r in data_records if r["_sidx"] not in processed_sidx]

print(f"Tìm thấy {len(failed_records)} mẫu bị kẹt:")
print("-" * 50)

# In thử 2 mẫu đầu tiên ra xem có gì bất thường không
for record in failed_records[:8]:
    subset = extract_subset_for_translation(record, COLUMNS_TO_TRANSLATE)
    print(f"Mẫu ID _sidx = {record['_sidx']}")
    print(json.dumps(subset, ensure_ascii=False, indent=2))
    print("-" * 50)

Tìm thấy 4 mẫu bị kẹt:
--------------------------------------------------
Mẫu ID _sidx = 96
{
  "QA_Type": "Disease Diagnosis",
  "question": "A 25-year-old female with Hodgkin's lymphoma presents with a several day history of edema. Lab studies show:\n\nSerum Na+: 140 mmol/L\nSerum K+: 3.5 mmol/L\nSerum albumin: 1.9 g/dL\nTotal serum bilirubin: 1.0 mg/dL\nSerum creatinine: 1.2 mg/dL\n\nUrinalysis shows 4+ proteinuria and fatty casts. What is the most likely diagnosis?\nA. Focal segmental glomerulosclerosis\nB. Membranous nephropathy\nC. Minimal change disease\nD. Amyloidosis\nE. Membranoproliferative glomerulonephritis",
  "answer": "C. Minimal change disease",
  "Scoring_Points": [
    "Patient presenting with edema, proteinuria, urine fatty casts, and hypoalbuminemia is suggesitve of nephrotic syndrome. ",
    "In a young paitent the most likely cause of nephrotic syndrome is minimal change disease. ",
    "There is an association between Hodgkins lymphoma and nephrotic syndrome. "


**Check lỗi 1 mẫu cụ thể**

In [ ]:
import aiohttp
import asyncio
import json

# Lấy lại data gốc để test mẫu 14
data_records = get_input_data()
if not data_records: data_records = [SAMPLE_TEXT]
for i, record in enumerate(data_records): record["_sidx"] = i

# Lấy đúng mẫu số 14
test_record = next((r for r in data_records if r["_sidx"] == 14), None)
subset_to_translate = extract_subset_for_translation(test_record, COLUMNS_TO_TRANSLATE)
source_json_str = json.dumps(subset_to_translate, ensure_ascii=False)

async def debug_raw_api():
    print("Đang gọi API cho mẫu số 14...")
    user_prompt = USER_PROMPT_DRAFT_TEMPLATE.format(json_data=source_json_str)
    
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": "openai/gpt-4o-mini", # Hoặc model em đang dùng
        "messages": [{"role": "system", "content": SYSTEM_PROMPT_DRAFT}, 
                     {"role": "user", "content": user_prompt}],
        "response_format": {"type": "json_object"}
    }

    async with aiohttp.ClientSession() as session:
        async with session.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload) as response:
            data = await response.json()
            print("\nToàn bộ cục dữ liệu API trả về:")
            print(data)
            try:
                content = data['choices'][0]['message']['content']
                print("\n" + "="*50)
                print("RAW TEXT TỪ MODEL TRẢ VỀ (CHƯA PARSE):")
                print("="*50)
                print(content)
                print("="*50)
                
                # Thử parse JSON xem lỗi chính xác là gì
                print("\nThử chạy json.loads()...")
                json.loads(content)
                print("Parse thành công! (Lạ nhỉ?)")
            except Exception as e:
                print(f"Lỗi Parse JSON: {e}")

# Chạy code debug
await debug_raw_api()